# Bibliotecas necessárias

In [1]:
import sys
from pyspark.sql import SparkSession
sys.path.append('/home/jovyan/work')
from datetime import datetime, timedelta
from crawler_sti import create_df
import os
from utils.spark.SendBuckets import SendBuckets

tabela = 'sti_atendimentos_sgt'
endpoint = 'https://apisgt.cni.org.br:8253/reports/v1.0.0/atendimentos'

# Cricação da variáveis necessárias

In [3]:
# Inicializa a sessão do Spark
spark = SparkSession.builder \
            .appName(f"STG to Bronze") \
            .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.12.571") \
            .config("spark.hadoop.fs.s3a.endpoint", os.getenv('MINIO_HOST')) \
            .config("spark.hadoop.fs.s3a.access.key", os.getenv('MINIO_ACCESS_KEY')) \
            .config("spark.hadoop.fs.s3a.secret.key", os.getenv('MINIO_SECRET_KEY')) \
            .config("spark.hadoop.fs.s3a.path.style.access", True) \
            .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem").getOrCreate()

#Bucket a ser gravado
send_buckets = SendBuckets('bronze',spark)

In [4]:
#Último x dias
#dias = 5
#start_date = datetime.now() - timedelta(days=dias)
#date = start_date.strftime('%Y%m%d')

# Data Atul
#date_now = datetime.now().strftime('%Y%m%d')

#D ata específica
date = '20230101'

# Extração

In [5]:
df = create_df(endpoint, date, tabela, 'full')

/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apisgt.cni.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Erro 502. Tentando novamente em 10 segundos...


/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apisgt.cni.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


DataFrame criado para sti_atendimentos_sgt na data 2024-04-17


# Cria tabela de particionamento por data (beta)

In [6]:
# Particionamento por ano, mes e dia contidos na própria tabela
#df['date_partition'] = pd.to_datetime(df['ano'].astype(str) + '-' + df['mes'].apply(lambda x: str(x).zfill(2)) + '-' + df['dia'].apply(lambda x: str(x).zfill(2)), format='%Y-%m-%d')

# Gravação no datalake

In [6]:
send_buckets.send_dataframe_bucket_part('overwrite', f'sti/apisgt/{tabela}', df, 'date_partition')

Salvando DataFrame em sti/apisgt/sti_atendimentos_sgt no MinIO
Removendo arquivos de partições correspondentes no MinIO...
Removendo arquivos de sti/apisgt/sti_atendimentos_sgt/date_partition=20240417
Envio concluído


In [7]:
spark.stop()